# `equity_risk_premium(country=None, on=None, revision=None)`

The total equity risk premium Damodaran estimates for each country, from `macro.damodaran_country_risk`: the edition's mature-market premium plus the country premium. Two estimates per row: `erp` (rating-based) and `cds_erp` (CDS-based, NULL where no CDS trades).

Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

`country` is an ISO alpha-3 code (`AE-AZ`, `AE-RK`, `AE-SH` for the three emirates) or a name from that edition, in any case and spacing; `None` gives every country. `on` picks the edition in force on that date (default: today in US/Eastern), and `revision` defaults to its newest. Values are Damodaran's, as published and never recomputed, in fractions: 0.0442 is 4.42%.

In [1]:
from datetime import date

import polars as pl
from IPython.display import Markdown

from arora_data.equity_risk_premium import EquityRiskPremium
from arora_data.sync import country_risk_premium, damodaran_editions, equity_risk_premium
from arora_data.dates import today_et
from arora_data.frames import to_df


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## One country, by code or name

In [2]:
assert equity_risk_premium("DEU") == equity_risk_premium("germany")
show(equity_risk_premium("DEU") + equity_risk_premium("IND") + equity_risk_premium("USA"))

3 rows

edition_date|revision|entity_code|published_name|entity_type|region|erp|cds_erp
---|---|---|---|---|---|---|---
2026-07-01|1|DEU|Germany|country|Western Europe|0.0420000000|0.0429326973
2026-07-01|1|IND|India|country|Asia|0.0692018253|0.0531923678
2026-07-01|1|USA|United States|country|North America|0.0442000000|0.0486843308

## erp is the mature premium plus crp
Checked on every row of today's edition. The one exception is the US, whose `erp` is his separate US premium (`us_erp`).

In [3]:
edition = [e for e in damodaran_editions() if e.latest][-1]
crp = {r.entity_code: r.crp for r in country_risk_premium()}
off = [r.entity_code for r in equity_risk_premium() if r.erp != edition.mature_erp + crp[r.entity_code]]
print("mature", edition.mature_erp, "| us_erp", edition.us_erp, "| rows that differ:", off)

mature 0.0420000000 | us_erp 0.0442000000 | rows that differ: ['USA']


## The whole edition, lowest premium first

In [4]:
erp = to_df(equity_risk_premium(), EquityRiskPremium)
erp.sort("erp").select("entity_code", "published_name", (pl.col("erp") * 100).alias("erp, %"), (pl.col("cds_erp") * 100).alias("cds_erp, %"))

entity_code,published_name,"erp, %","cds_erp, %"
str,str,"decimal[38,10]","decimal[38,10]"
"""AUS""","""Australia""",4.2000000000,4.3088146900
"""CAN""","""Canada""",4.2000000000,4.4487192800
"""CHE""","""Switzerland""",4.2000000000,4.2000000000
"""DEU""","""Germany""",4.2000000000,4.2932697300
"""DNK""","""Denmark""",4.2000000000,4.2932697300
…,…,…,…
"""YEM""","""Yemen, Republic""",19.0519057100,null
"""BLR""","""Belarus""",31.4036717100,null
"""BRB""","""Barbados""",31.4036717100,null


## The US across editions
From 2026 his file carries a separate US premium.

In [5]:
for e in damodaran_editions():
    if e.latest:
        r = equity_risk_premium("USA", on=e.edition_date)[0]
        print(r.edition_date, "erp", f"{r.erp * 100:.2f}%", "| mature", f"{e.mature_erp * 100:.2f}%")

2015-01-01 erp 5.75% | mature 5.75%


2016-07-01 erp 6.25% | mature 6.25%


2017-01-05 erp 5.69% | mature 5.69%


2018-01-01 erp 5.08% | mature 5.08%


2019-01-01 erp 5.96% | mature 5.96%


2020-01-01 erp 5.20% | mature 5.20%


... (5 outputs left out)

2025-07-01 erp 4.62% | mature 4.21%


2026-01-01 erp 4.46% | mature 4.23%


2026-04-01 erp 5.03% | mature 4.77%


2026-07-01 erp 4.42% | mature 4.20%


## The 2017-01-05 edition's PRS countries
For countries without a Moody's rating, his 2017 file gives an `erp` that isn't the mature premium plus `crp`. Served as his file has it.

In [6]:
on = date(2017, 1, 5)
mature = [e for e in damodaran_editions() if e.edition_date == on][0].mature_erp
for c in ["GIN", "LBY", "ZWE"]:
    print(c, "erp", equity_risk_premium(c, on)[0].erp, "| mature + crp", mature + country_risk_premium(c, on)[0].crp)

GIN erp 0.2000000000 | mature + crp 0.1777488908


LBY erp 0.2000000000 | mature + crp 0.1777488908


ZWE erp 0.1724000000 | mature + crp 0.1534912226


## Known source issue
In the 2026-07-01 edition Barbados's Moody's rating is typed as C in his file, so its `erp` is Belarus's, 31.40%.

In [7]:
show(equity_risk_premium("BRB", on=date(2026, 7, 1)) + equity_risk_premium("BLR", on=date(2026, 7, 1)))

2 rows

edition_date|revision|entity_code|published_name|entity_type|region|erp|cds_erp
---|---|---|---|---|---|---|---
2026-07-01|1|BRB|Barbados|country|Caribbean|0.3140367171|None
2026-07-01|1|BLR|Belarus|country|Eastern Europe & Russia|0.3140367171|None

## Bad input

In [8]:
for kwargs in [dict(country="South Korea"), dict(country="US"), dict(on=date(2014, 12, 31)),
               dict(on=date(2026, 3, 31), revision=7)]:
    try:
        equity_risk_premium(**kwargs)
    except ValueError as e:
        print(e)

'South Korea' is not a stored country in Damodaran's 2026-07-01 edition: use an ISO alpha-3 code or a name from that edition, such as USA or 'United States'


'US' is not a stored country in Damodaran's 2026-07-01 edition: use an ISO alpha-3 code or a name from that edition, such as USA or 'United States'


no Damodaran edition on or before 2014-12-31: the first is 2015-01-01


Damodaran's 2026-01-01 edition has no revision 7: it has 1, 2, 3, 4
